In [0]:
library(dataiku)
library(dplyr)

In [0]:
# Recipe inputs
base_data_regions <- dkuReadDataset("base_data_regions", samplingMethod="head", nbRows=100000)

In [0]:
colnames(base_data_regions)

In [0]:
# Creating a function that generates a counterfactual dataset

#' @title counterfactual_gen
#' @description Function takes argguments of df, tc, matches and returns a
#' counterfactual dataframe to be used by the hurdle function
#' @param df counterfactual_test_data
#' @param tc tropical cyclone name
#' @param matches municipality matches NOT NEEDED
#' @return counterfactual_data_list return

counterfactual_gen  <- function(df, tc, storm_surge, landslide){
    # function requires dplyr to run correctly
    library(dplyr)

    # get unique municipality codes
    mun_code  <- unique(df$Mun_Code)

    # filter df by tc and get hazard characheristics
    counterfactual_data  <- df %>%
        filter(typhoon == tc) %>% # keep the minimum distance from the filter
        mutate(
              # Primary hazard measurements are based on the closest path to land/municipality.
              # ie where was haddest hit
              HAZ_rainfall_Total = HAZ_rainfall_Total[which.min(HAZ_dis_track_min)],
              HAZ_rainfall_max_6h = HAZ_rainfall_max_6h[which.min(HAZ_dis_track_min)],
              HAZ_rainfall_max_24h = HAZ_rainfall_max_24h[which.min(HAZ_dis_track_min)],
              HAZ_v_max = HAZ_v_max[which.min(HAZ_dis_track_min)],
              HAZ_dis_track_min = min(HAZ_dis_track_min, na.rm = TRUE),
              # setting secondary hard to storm_surge value or landslide value specified in the function call
              GEN_landslide_per = ,
              GEN_stormsurge_per = ,
              GEN_Bu_p_inSSA = ,
              GEN_Bu_p_LS = ,
              GEN_Red_per_LSbldg = ,
              GEN_Or_per_LSblg = ,
              GEN_Yel_per_LSSAb = ,
              GEN_RED_per_SSAbldg = ,
              GEN_OR_per_SSAbldg = ,
              GEN_Yellow_per_LSbl = ,
              TOP_mean_slope = ,
              TOP_mean_elevation_m = ,
              TOP_ruggedness_stdev = ,
              TOP_mean_ruggedness = ,
              TOP_slope_stdev = ,
              VUL_poverty_perc = ,
              GEN_with_coast = , 
              GEN_coast_length , 
              VUL_Housing_Units ,
              VUL_StrongRoof_StrongWall ,
              VUL_StrongRoof_LightWall ,
              VUL_StrongRoof_SalvageWall, 
              VUL_LightRoof_StrongWall, 
              VUL_LightRoof_LightWall, 
              VUL_LightRoof_SalvageWall, 
              VUL_SalvagedRoof_StrongWall,
              VUL_SalvagedRoof_LightWall,
              VUL_SalvagedRoof_SalvageWall, 
              VUL_vulnerable_groups, 
              VUL_pantawid_pamilya_beneficiary
              ) %>%
        select(-typhoon)

    # which municipalities are not in the filtered data?
    missing_mun  <- setdiff(mun_code, counterfactual_data$Mun_Code)

    # debugging
    #cat("number of missing municipalities:", sep = " ", length(missing_mun))

    # Check if there are any missing municipalities
    if (length(missing_mun) > 0) {
        # Get the characteristics of the missing mun codes
        #remaining_mun <- df %>%
        #    filter(Mun_Code %in% missing_mun) %>%
        #    select(-typhoon, -rain_total, -wind_max, -track_min_dist)

        # Assign the hazard characteristics from the counterfactual data
        remaining_mun <- df %>%
            filter(Mun_Code %in% missing_mun) %>% # after filtering Mun_Code has duplicates how do we remove duplicates?
            distinct(Mun_Code, .keep_all = TRUE) %>%  # Keeps the first occurrence of each Mun_Code
            mutate(rain_total = unique(counterfactual_data$rain_total),
                   wind_max = unique(counterfactual_data$wind_max),
                   wind_blue_ss = wind_max * storm_surge,
                   wind_yellow_ss = wind_max * storm_surge,
                   wind_orange_ss = wind_max * storm_surge,
                   wind_red_ss = wind_max * storm_surge,
                   rain_blue_ss = rain_total * landslide,
                   rain_yellow_ss = rain_total * landslide,
                   rain_orange_ss = rain_total * landslide,
                   rain_red_ss = rain_total * landslide,
                   damage_perc = 0, # set damage variable to zero or "Damage_below_10"
                   damage_binary = 0,
                   damage_binary_2 = "Damage_below_10"
                  ) %>%
        select(-typhoon)
        # debugging
        cat("number of columns in remaining_mun", sep = " ", ncol(remaining_mun))

        cat("\n number of columns in counterfactual data", sep = " ", ncol(counterfactual_data))

        # Add the remaining municipalities back into the counterfactual data
        counterfactual_data <- rbind(counterfactual_data, remaining_mun)
    }

    # df should have all the 1478 municipalities

    return(counterfactual_data) # returns a dataframe (maybe list for more experiments)
}

In [0]:
# Recipe outputs
v510_counterfactual_datasets_fixed_sec_hazards <- dkuManagedFolderPath("F8NXOAoc")